# MediGuide — QLoRA Fine-Tuning
### Qwen2.5-1.5B-Instruct + 4-bit NF4 base + LoRA on Doctor-Patient Conversations

Same LoRA adapter setup as the previous notebook (`r=16, alpha=32, dropout=0.05`, attention+MLP
target modules), but the **frozen base model is loaded in 4-bit (NF4)** via `bitsandbytes`
instead of fp16. This is the standard QLoRA setup (Dettmers et al., 2023): base weights sit in
4-bit and dequantize on-the-fly to fp16 for each matmul; only the LoRA adapter trains, in fp16,
on top.

**Not the same thing as MXFP4/BRQ-style post-training quantization** — that's a different
technique (PTQ for inference, needs Blackwell-class hardware) that doesn't apply here. See
discussion earlier in this project; T4 doesn't support MXFP4 natively.

**What we expect to see, going in** (worth checking against the real numbers, not assuming):
- **Size**: base model footprint should drop from ~3GB (fp16) to ~0.8-0.9GB (NF4) — the LoRA
  adapter itself stays about the same size (~70MB) regardless of base precision.
- **Quality**: original QLoRA results were mostly demonstrated on much larger models (13B-65B).
  At 1.5B there's less redundancy to absorb quantization noise, so ROUGE/BLEU/perplexity may land
  close to, or slightly below, the fp16 LoRA run's numbers (rougeL 0.119, ppl 14.5) — that gap
  itself is a legitimate finding, not a failure.
- **Latency**: NF4 dequantization happens at every matmul, so generation is expected to be
  at-or-above the fp16 LoRA latency (~2.34s/example), not below it — smaller footprint, not
  necessarily faster.

Same seed (8), same tokenizer (unmodified), same prompt template, same eval pipeline as the
previous two notebooks, so all three summaries can be joined directly for the final report.


## 1. Setup

In [ ]:
!pip install -q -U peft accelerate bitsandbytes evaluate rouge_score sacrebleu sentencepiece
!pip uninstall -y torchao -q


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single GPU only — bitsandbytes 4-bit layers are not
                                            # safe under Trainer's automatic DataParallel wrapping,
                                            # which kicks in whenever >1 GPU is visible and caused
                                            # the "illegal memory access" in bnb's gemm_4bit op.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # fragmentation guard, must be set before torch initializes CUDA

import json, time, math, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
DATA_DIR = "/kaggle/input/datasets/totaldose/doctor-patient-conversation/data/processed"

def find_split_file(data_dir, split_names):
    for name in split_names:
        p = os.path.join(data_dir, f"{name}.json")
        if os.path.exists(p):
            return p
    return None

TRAIN_PATH = find_split_file(DATA_DIR, ["train"])
VAL_PATH   = find_split_file(DATA_DIR, ["val", "valid", "validation", "dev"])
TEST_PATH  = find_split_file(DATA_DIR, ["test"])

print("train:", TRAIN_PATH)
print("val:  ", VAL_PATH)
print("test: ", TEST_PATH)
assert TRAIN_PATH is not None and TEST_PATH is not None, "Check DATA_DIR"


In [ ]:
# Prior results, for the final comparison table
RESULT_CANDIDATES = {
    "baseline": ["/kaggle/working/results/baseline_summary.json",
                 "/kaggle/input/mediguide-baseline/baseline_summary.json"],
    "lora": ["/kaggle/working/results/lora_summary.json",
             "/kaggle/input/mediguide-lora/lora_summary.json"],
}
prior_summaries = {}
for name, candidates in RESULT_CANDIDATES.items():
    path = next((p for p in candidates if os.path.exists(p)), None)
    if path:
        with open(path) as f:
            prior_summaries[name] = json.load(f)
        print(f"Found {name} summary at {path}")
    else:
        print(f"WARNING: {name}_summary.json not found automatically — set its path manually "
              f"below if you want it in the final comparison table.")


## 2. Tokenizer — unmodified, sanity-checked

Same check as the previous two notebooks. Quantizing the *base weights* has nothing to do with
the tokenizer/vocab — we still never call `add_tokens` or `resize_token_embeddings`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)

print("len(tokenizer):    ", len(tokenizer))
print("config.vocab_size: ", config.vocab_size)
assert config.vocab_size >= len(tokenizer)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right-padding for training


## 3. Load base model in 4-bit (NF4) and apply LoRA

`prepare_model_for_kbit_training` handles what quantized models need before backprop works
correctly: casts norm layers to fp32 for stability, sets up gradient checkpointing properly, and
enables grad on the input embeddings (needed since the quantized base has no trainable params of
its own for autograd to hook into otherwise).


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,       # quantizes the quantization constants too, small extra savings
    bnb_4bit_compute_dtype=torch.float16, # T4 has no bf16 tensor cores, use fp16 for the actual matmuls
)

torch.cuda.reset_peak_memory_stats()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)

base_model_mem_gb = torch.cuda.memory_allocated() / (1024**3)
print(f"GPU memory after loading 4-bit base model: {base_model_mem_gb:.3f} GB")

base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
base_model.config.use_cache = False


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],  # attention + MLP, same as fp16 LoRA run
    bias="none",
    # modules_to_save intentionally unset — same reasoning as the fp16 LoRA notebook
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


## 4. Dataset preparation (identical to the fp16 LoRA notebook)

In [ ]:
SYSTEM_PROMPT = (
    "You are a medical information assistant. Respond to the patient's question with "
    "clear, professional, and clinically sound guidance, consistent with recognized clinical "
    "guidelines. Use formal medical language appropriate for a patient audience. Always make "
    "clear that your response is informational only and does not replace an in-person "
    "diagnosis or professional medical care."
)

MAX_PROMPT_LENGTH = 768
MAX_TARGET_LENGTH = 384   # matches the value the fp16 LoRA run settled on after the OOM fix
MAX_TOTAL_LENGTH = MAX_PROMPT_LENGTH + MAX_TARGET_LENGTH

def build_prompt(description, patient, for_generation=True):
    user_turn = f"{description.strip()}\n\n{patient.strip()}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_turn},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=for_generation
    )

def encode_example(description, patient, doctor):
    prompt_text = build_prompt(description, patient, for_generation=True)
    prompt_ids = tokenizer(
        prompt_text, truncation=True, max_length=MAX_PROMPT_LENGTH, add_special_tokens=False
    )["input_ids"]
    target_ids = tokenizer(
        doctor.strip() + tokenizer.eos_token,
        truncation=True, max_length=MAX_TARGET_LENGTH, add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


In [ ]:
class DoctorPatientDataset(Dataset):
    def __init__(self, df):
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        return encode_example(row["Description"], row["Patient"], row["Doctor"])


def load_split(path):
    with open(path) as f:
        return pd.DataFrame(json.load(f))

df_train = load_split(TRAIN_PATH)
df_val = load_split(VAL_PATH) if VAL_PATH else None
df_test = load_split(TEST_PATH)

print("train:", len(df_train))
if df_val is not None:
    print("val:  ", len(df_val))
print("test: ", len(df_test))

train_dataset = DoctorPatientDataset(df_train)
eval_dataset = DoctorPatientDataset(df_val) if df_val is not None else None

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100,
)


## 5. Training configuration

Starting from `per_device_train_batch_size=2` rather than the `1` the fp16 LoRA run needed —
the 4-bit base frees up real VRAM, so we should be able to afford a bit more headroom. If this
still OOMs, drop to 1 and raise `gradient_accumulation_steps` to compensate (same fix as before).
We track **peak GPU memory** explicitly since that's QLoRA's actual selling point and needs a
real number, not an assumption.


In [ ]:
OUTPUT_DIR = "/kaggle/working/qlora_checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch size 16, same as fp16 LoRA run
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    fp16=True,
    logging_steps=20,
    eval_strategy="epoch" if eval_dataset is not None else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=eval_dataset is not None,
    metric_for_best_model="eval_loss" if eval_dataset is not None else None,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)


In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_result = trainer.train()

train_wall_seconds = time.time() - t0
peak_train_mem_gb = torch.cuda.max_memory_allocated() / (1024**3)

print(train_result)
print(f"\nTraining wall-clock time: {train_wall_seconds:.1f}s")
print(f"Peak GPU memory during training: {peak_train_mem_gb:.3f} GB")


## 6. Save the adapter

Adapter size should land close to the fp16 LoRA run's ~70MB — quantizing the *frozen base* does
not change the adapter's own size, since the adapter itself is still stored in fp16/fp32.


In [ ]:
ADAPTER_DIR = "/kaggle/working/qlora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_file = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
if os.path.exists(adapter_file):
    adapter_size_mb = os.path.getsize(adapter_file) / (1024 * 1024)
    print(f"Adapter size: {adapter_size_mb:.1f} MB")
else:
    adapter_size_mb = None
    print("adapter_model.safetensors not found — check ADAPTER_DIR contents:", os.listdir(ADAPTER_DIR))

with open(os.path.join(ADAPTER_DIR, "adapter_config.json")) as f:
    adapter_cfg = json.load(f)
print("modules_to_save:", adapter_cfg.get("modules_to_save"))
assert not adapter_cfg.get("modules_to_save")


## 7. Re-run the same evaluation pipeline

Identical to the previous two notebooks: greedy decoding, same `MAX_NEW_TOKENS`, teacher-forced
perplexity, ROUGE-L/BLEU. `model` here is the 4-bit-base + LoRA-adapter PeftModel from training.


In [ ]:
model.eval()
model.config.use_cache = True
tokenizer.padding_side = "left"  # required for batched generation

GEN_MAX_NEW_TOKENS = 256
GEN_BATCH_SIZE = 8

def generate_batch(prompts, max_new_tokens=GEN_MAX_NEW_TOKENS):
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_PROMPT_LENGTH
    ).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts], elapsed / len(prompts)


In [ ]:
prompts = [build_prompt(row["Description"], row["Patient"]) for _, row in df_test.iterrows()]

predictions, gen_seconds = [], []
t0 = time.time()
for i in range(0, len(prompts), GEN_BATCH_SIZE):
    batch = prompts[i:i + GEN_BATCH_SIZE]
    texts, per_ex_time = generate_batch(batch)
    predictions.extend(texts)
    gen_seconds.extend([per_ex_time] * len(texts))
    print(f"{min(i + GEN_BATCH_SIZE, len(prompts))}/{len(prompts)} done, "
          f"{time.time() - t0:.1f}s elapsed", flush=True)

df_test["prediction"] = predictions
df_test["gen_seconds"] = gen_seconds
total_gen_time = time.time() - t0
print(f"Total generation time: {total_gen_time:.1f}s")


In [ ]:
def compute_example_loss(prompt, target):
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_PROMPT_LENGTH,
                            add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False,
                            truncation=True, max_length=MAX_TARGET_LENGTH)["input_ids"]
    input_ids = torch.tensor([prompt_ids + target_ids]).to(DEVICE)
    labels = input_ids.clone()
    labels[:, :len(prompt_ids)] = -100
    with torch.no_grad():
        out = model(input_ids=input_ids, labels=labels)
    return out.loss.item(), len(target_ids)


In [ ]:
losses, n_target_tokens = [], []
t0 = time.time()
for i, row in df_test.iterrows():
    prompt = build_prompt(row["Description"], row["Patient"])
    loss, n_tok = compute_example_loss(prompt, row["Doctor"])
    losses.append(loss)
    n_target_tokens.append(n_tok)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(df_test)} scored, {time.time() - t0:.1f}s elapsed", flush=True)

df_test["loss"] = losses
df_test["n_target_tokens"] = n_target_tokens

total_loss_tokens = (df_test["loss"] * df_test["n_target_tokens"]).sum()
total_tokens = df_test["n_target_tokens"].sum()
corpus_ppl = math.exp(total_loss_tokens / total_tokens)
df_test["perplexity"] = df_test["loss"].apply(math.exp)
print(f"Corpus-level perplexity (QLoRA): {corpus_ppl:.3f}")


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

references = df_test["Doctor"].tolist()
rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
bleu_scores = bleu.compute(predictions=predictions, references=[[r] for r in references])

rouge_per_row = [rouge.compute(predictions=[p], references=[r], use_stemmer=True)["rougeL"]
                  for p, r in zip(predictions, references)]
df_test["rougeL"] = rouge_per_row

print("ROUGE:", rouge_scores)
print("BLEU: ", bleu_scores["score"])


## 8. QLoRA results summary + full comparison

In [ ]:
qlora_summary = {
    "model": MODEL_NAME + " + QLoRA (NF4)",
    "seed": SEED,
    "n_test_examples": len(df_test),
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "bleu": bleu_scores["score"],
    "perplexity_corpus": corpus_ppl,
    "avg_generation_seconds_per_example": df_test["gen_seconds"].mean(),
    "max_new_tokens": GEN_MAX_NEW_TOKENS,
    "total_wall_clock_generation_seconds": total_gen_time,
    "adapter_size_mb": adapter_size_mb,
    "base_model_gpu_mem_gb_4bit": base_model_mem_gb,
    "peak_training_gpu_mem_gb": peak_train_mem_gb,
    "training_wall_clock_seconds": train_wall_seconds,
}

os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/qlora_summary.json", "w") as f:
    json.dump(qlora_summary, f, indent=2)
df_test.to_csv("/kaggle/working/results/qlora_predictions.csv", index=False)

pd.DataFrame([qlora_summary]).T.rename(columns={0: "value"})


In [ ]:
# Three-way comparison, if baseline and fp16-LoRA summaries were found earlier
all_summaries = {**prior_summaries, "qlora": qlora_summary}
compare_keys = ["rouge1", "rouge2", "rougeL", "bleu", "perplexity_corpus",
                 "avg_generation_seconds_per_example", "adapter_size_mb"]

if len(all_summaries) > 1:
    comparison = pd.DataFrame({
        name: {k: s.get(k) for k in compare_keys} for name, s in all_summaries.items()
    })
    display(comparison.round(4))
else:
    print("Only qlora_summary available in this session — load baseline_summary.json and "
          "lora_summary.json (set their paths in RESULT_CANDIDATES above) for the full table.")


In [ ]:
# Stratified by severity, same as the previous two notebooks
strat = df_test.groupby("Status").agg(
    n=("Doctor", "count"),
    rougeL=("rougeL", "mean"),
    perplexity=("perplexity", "mean"),
).round(4)
strat


In [ ]:
for i in df_test.sample(3, random_state=SEED).index:
    row = df_test.loc[i]
    print("="*100)
    print("SEVERITY:", row["Status"])
    print("PATIENT :", row["Patient"][:250], "...")
    print("-"*100)
    print("REFERENCE DOCTOR:", row["Doctor"][:400])
    print("-"*100)
    print("QLORA PREDICTION:", row["prediction"][:400])


## 9. Next steps

- `results/qlora_summary.json` and `results/qlora_predictions.csv` saved for the final report.
- `qlora_adapter/` holds the trained adapter.
- Compare `base_model_gpu_mem_gb_4bit` here against the fp16 LoRA run's base model footprint
  (~3GB) to quantify the actual memory savings — this is the number that matters most for the
  report's "resource utilization" column, more than the adapter size (which barely changes).
- Remaining: Prompt Tuning run, then the full comparison table + PDF report.
